# 060 — Why Simple RNNs Fail

BPTT is correct and simple RNNs still fail. The reason is in lesson 059's
formula: the gradient reaching $t$ steps back carries a **product of $t$
matrices**. Measured here on a 100-step sequence at an entirely ordinary
spectral radius of 0.9, the gradient 99 steps back is **5.1e-26** and the
effective memory is **9 steps**.

| Part | What we check |
|---|---|
| A | gradient against distance, at four spectral radii |
| B | effective memory at the 1% threshold |
| C | the mirror failure: ReLU and large weights explode |
| D | clipping, and what it does and does not change |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — How far back does the gradient reach?

In [ ]:
import numpy as np

HID, FEAT = 20, 8

def gradient_flow(steps, radius, seed=0, relu=False):
    """Norm of dL/dh_t as t moves away from the loss."""
    r = np.random.default_rng(seed)
    Wh = r.normal(0, 1, (HID, HID))
    Wh *= radius / max(abs(np.linalg.eigvals(Wh)))      # set the spectral radius
    Wx, Wo = r.normal(0, 0.3, (FEAT, HID)), r.normal(0, 0.3, HID)
    X = r.normal(0, 1, (steps, FEAT))
    activate = (lambda z: np.maximum(0, z)) if relu else np.tanh

    hs = [np.zeros(HID)]
    for t in range(steps):
        hs.append(activate(X[t] @ Wx + hs[-1] @ Wh))

    dh, norms = Wo.copy(), []
    for t in range(steps - 1, -1, -1):
        norms.append(np.linalg.norm(dh))
        dh = Wh @ (dh * ((hs[t + 1] > 0) if relu else (1 - hs[t + 1] ** 2)))
    return np.array(norms[::-1])

RADII = (0.5, 0.9, 1.0, 1.5)
flows = {r: gradient_flow(100, r) for r in RADII}
print(f"  {'steps back':>11}" + "".join(f"{f'rho={r}':>13}" for r in RADII))
for back in (0, 1, 5, 10, 20, 50, 99):
    print(f"  {back:>11}" + "".join(f"{flows[r][-1 - back]:>13.2e}" for r in RADII))
assert f"{flows[0.9][0]:.2e}" == "5.13e-26"
print("\nEven at rho = 1.0 it vanishes, because tanh' <= 1 is also a factor")

## Part B — Effective memory

Take the point where the gradient has fallen below 1% of its starting size.

In [ ]:
def memory(flow):
    relative = flow[::-1] / flow[-1]
    below = np.where(relative < 0.01)[0]
    return int(below[0]) if len(below) else None

rows = []
for radius in (0.5, 0.9, 1.0, 1.2, 1.5):
    flow = flows.get(radius) if radius in flows else gradient_flow(100, radius)
    rows.append((radius, memory(flow), flow[0]))
    print(f"  spectral radius {radius}: {memory(flow):>3} steps   "
          f"(gradient 99 back {flow[0]:.2e})")
assert dict((r, m) for r, m, _ in rows)[0.9] == 9
assert dict((r, m) for r, m, _ in rows)[1.0] == 10
print()
print("Nine steps at 0.9, ten at 1.0. The folklore figure - 'an RNN remembers")
print("about ten words' - arrived at independently, as a number.")

## Part C — The mirror failure

In [ ]:
for radius in (1.0, 1.5, 2.0):
    flow = gradient_flow(100, radius, relu=True)
    flag = "   <- exploded" if flow[0] > 1e6 else ""
    print(f"  relu, rho={radius}: gradient 99 steps back = {flow[0]:.3e}{flag}")
exploded = gradient_flow(100, 2.0, relu=True)[0]
assert exploded > 1e5
print()
print("Vanishing and exploding are ONE phenomenon with the exponent's sign")
print("flipped: base < 1 shrinks, base > 1 grows. tanh caps the base below 1,")
print("which is why tanh RNNs vanish and never explode.")

## Part D — Clipping

In [ ]:
norm = exploded
for threshold in (1.0, 5.0):
    scale = min(1.0, threshold / norm)
    print(f"  norm {norm:.3e} -> {min(norm, threshold):.1f}   "
          f"(scaled by {scale:.2e})")
    assert scale < 1

# clipping rescales, it does not rotate: the direction is untouched
g = np.array([3.0, -4.0])
clipped = g * min(1.0, 1.0 / np.linalg.norm(g))
print(f"\n  {g} -> {clipped.round(3)}   norm {np.linalg.norm(clipped):.3f}")
assert np.allclose(clipped / np.linalg.norm(clipped), g / np.linalg.norm(g))
print("  same direction, length 1 - clipping fixes exploding and does NOTHING")
print("  for vanishing, which is the problem that needed a new architecture.")

## What this notebook established

- The gradient reaching $t$ steps back is a product of $t$ matrices, so it is
  exponential in $t$: **5.1e-26** at 99 steps back with $\rho = 0.9$.
- Effective memory at the 1% threshold: **9 steps** at $\rho = 0.9$, **10** at
  $\rho = 1.0$, 16 at 1.5.
- It vanishes even at $\rho = 1.0$, because $\tanh' \le 1$ multiplies in too.
- With ReLU and $\rho = 2.0$ the same quantity reaches **2.9e+05** — the same
  phenomenon with the exponent's sign flipped.
- Clipping rescales without rotating, so it cures exploding and does nothing at
  all for vanishing.

## Exercises

1. Find the spectral radius at which the effective memory first passes 30
   steps. What has happened to the forward activations by then?
2. Plot the four flows on a log axis. Which is a straight line, and why?
3. Clip at 0.1 and re-run Part C. Does the direction survive?